<a href="https://colab.research.google.com/github/NeuronEdge67/Hands-On-Machine-Learning-...By-Aur-lien-G-ron-/blob/main/10_neural_nets_with_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [74]:
import torch

In [75]:
torch.set_default_device('cuda')

In [76]:
X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]])
X

tensor([[1., 4., 7.],
        [2., 3., 6.]], device='cuda:0')

In [77]:
X.shape

torch.Size([2, 3])

In [78]:
X.dtype

torch.float32

In [79]:
a = X.device
a

device(type='cuda', index=0)

In [80]:
x = torch.tensor(6.0, requires_grad=True)
f = x ** 2
f

tensor(36., device='cuda:0', grad_fn=<PowBackward0>)

In [81]:
f.backward(retain_graph=True)
x.grad

tensor(12., device='cuda:0')

In [82]:
lr = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iter in range(100):
  f = x**2
  f.backward()
  with torch.no_grad():
    x -= lr * x.grad
  x.grad.zero_()

In [83]:
x

tensor(1.0185e-09, device='cuda:0', requires_grad=True)

In [84]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [85]:
housing = fetch_california_housing()
X_train, X_temp, y_train, y_temp = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42)

In [86]:
X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

In [87]:
y_train = torch.FloatTensor(y_train).view(-1, 1)
y_valid = torch.FloatTensor(y_valid).view(-1, 1)
y_test = torch.FloatTensor(y_test).view(-1, 1)

In [88]:
torch.manual_seed(42)
n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [89]:
learning_rate = 0.4
n_epochs = 20

X_train = X_train.to('cuda')
y_train = y_train.to('cuda')
w = w.to('cuda')
b = b.to('cuda')

for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

Epoch 1/20, Loss: 16.47666358947754
Epoch 2/20, Loss: 1.2107000350952148
Epoch 3/20, Loss: 0.6819401383399963
Epoch 4/20, Loss: 0.5730571746826172
Epoch 5/20, Loss: 0.5387604236602783
Epoch 6/20, Loss: 0.5267452001571655
Epoch 7/20, Loss: 0.5223101377487183
Epoch 8/20, Loss: 0.5205764174461365
Epoch 9/20, Loss: 0.5198367834091187
Epoch 10/20, Loss: 0.5194748640060425
Epoch 11/20, Loss: 0.519262969493866
Epoch 12/20, Loss: 0.519115149974823
Epoch 13/20, Loss: 0.5189981460571289
Epoch 14/20, Loss: 0.5188987851142883
Epoch 15/20, Loss: 0.5188113451004028
Epoch 16/20, Loss: 0.5187333226203918
Epoch 17/20, Loss: 0.5186628699302673
Epoch 18/20, Loss: 0.518599271774292
Epoch 19/20, Loss: 0.5185415744781494
Epoch 20/20, Loss: 0.5184893012046814


In [90]:
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)

In [91]:
print(model.bias)
print(model.weight)

Parameter containing:
tensor([0.3449], device='cuda:0', requires_grad=True)
Parameter containing:
tensor([[ 0.0799, -0.3464, -0.0718, -0.3251, -0.2431, -0.0124,  0.1671, -0.0665]],
       device='cuda:0', requires_grad=True)


In [92]:
model(X_train[:2])

tensor([[-0.2229],
        [-0.3159]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [93]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
 for epoch in range(n_epochs):
  y_pred = model(X_train)
  loss = criterion(y_pred, y_train)
  loss.backward()
  optimizer.step()
  optimizer.zero_grad()
  print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

In [94]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)


NameError: name 'optimizer' is not defined